In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Safe catalog/schema fallback
catalog = (
    os.getenv("DATABRICKS_BUNDLE_VAR_catalog") or
    os.getenv("catalog") or
    "hive_metastore"
)
schema = (
    os.getenv("DATABRICKS_BUNDLE_VAR_schema") or
    os.getenv("schema") or
    "default"
)

if not catalog or catalog.lower() in ["none", "null", ""]:
    raise ValueError("❌ No catalog provided. Check databricks.yml target overrides.")

input_table = f"{catalog}.{schema}.demo_sales"
output_table = f"{catalog}.{schema}.etl_demo_output"

print(f"Reading input table: {input_table}")
df = spark.read.table(input_table)

print("Adding amount_with_tax column...")
df_transformed = df.withColumn("amount_with_tax", df.amount * 1.2)

print(f"Writing transformed table: {output_table}")
df_transformed.write.mode("overwrite").saveAsTable(output_table)

print(f"✅ ETL demo table created at {output_table}")

In [ ]:
# Verify output
display(spark.table(output_table))